# Script 12B — Regime-Robust Volatility Ensemble

This notebook stabilizes Script 12's GARCH-family volatility forecasts without changing the underlying model family.

It uses:
- GARCH / EGARCH / GJR-GARCH forecasts already produced by Script 12
- EWMA, HIST20, HIST60 as observable volatility anchors
- previous-year-only strategy selection
- a past-only stress regime signal
- two-sided guardrails against near-zero and explosive variance forecasts
- QLIKE as the primary loss
- Kupiec unconditional coverage for H1 VaR

2024–2025 remains locked.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import t as student_t

warnings.filterwarnings("ignore")

CURRENCIES=["USD","GBP","EUR","JPY"]
HORIZONS=[1,3,7]
YEARS=[2019,2020,2021,2022,2023]
EPS=1e-10

MODEL_WEIGHTS=[0.25,0.50,0.75,1.00]
STRESS_THRESHOLDS=[1.5,2.0,3.0]
STRESS_MODEL_WEIGHT_CAPS=[0.25,0.50]

LOWER_MULT=0.10
UPPER_MULT=10.0

In [ ]:
def find_root():
    for root in [Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]:
        cand=root/"outputs"/"predictions"/"12_volatility_candidate_predictions_2018_2023.csv"
        dev=root/"data"/"processed"/"02_development_model_dataset.csv"
        if cand.exists() and dev.exists():
            return root.resolve()
    raise FileNotFoundError("Required Script 12 candidate predictions or Script 02 development data not found.")

ROOT=find_root()
CAND_PATH=ROOT/"outputs"/"predictions"/"12_volatility_candidate_predictions_2018_2023.csv"
DEV_PATH=ROOT/"data"/"processed"/"02_development_model_dataset.csv"
PRED_DIR=ROOT/"outputs"/"predictions"
TABLE_DIR=ROOT/"outputs"/"tables"
PRED_DIR.mkdir(parents=True,exist_ok=True)
TABLE_DIR.mkdir(parents=True,exist_ok=True)

cand=pd.read_csv(CAND_PATH,parse_dates=["Forecast_Origin"])
df=pd.read_csv(DEV_PATH,parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

assert df["Date"].max() < pd.Timestamp("2024-01-01")
assert cand["Forecast_Origin"].max() < pd.Timestamp("2024-01-01")

print("Root:",ROOT)
print("Models:",sorted(cand["Model"].dropna().unique()))
print("PASS — 2024–2025 untouched.")

In [ ]:
def qlike_losses(rv,vf):
    rv=np.maximum(np.asarray(rv,float),EPS)
    vf=np.maximum(np.asarray(vf,float),EPS)
    return np.log(vf)+rv/vf

def qlike(rv,vf):
    rv=np.asarray(rv,float); vf=np.asarray(vf,float)
    m=np.isfinite(rv)&np.isfinite(vf)&(vf>0)
    if not m.any(): return np.nan
    return float(np.mean(qlike_losses(rv[m],vf[m])))

def variance_mse(rv,vf):
    rv=np.asarray(rv,float); vf=np.asarray(vf,float)
    m=np.isfinite(rv)&np.isfinite(vf)
    return float(np.mean((rv[m]-vf[m])**2)) if m.any() else np.nan

def vol_rmse(rv,vf):
    rv=np.asarray(rv,float); vf=np.asarray(vf,float)
    m=np.isfinite(rv)&np.isfinite(vf)
    if not m.any(): return np.nan
    return float(np.sqrt(np.mean((np.sqrt(np.maximum(rv[m],0))-np.sqrt(np.maximum(vf[m],0)))**2)))

def dm_qlike(rv,model_v,bench_v,h):
    rv=np.asarray(rv,float); model_v=np.asarray(model_v,float); bench_v=np.asarray(bench_v,float)
    m=np.isfinite(rv)&np.isfinite(model_v)&np.isfinite(bench_v)&(model_v>0)&(bench_v>0)
    rv=rv[m]; model_v=model_v[m]; bench_v=bench_v[m]
    n=len(rv)
    if n<30: return np.nan,np.nan
    d=qlike_losses(rv,bench_v)-qlike_losses(rv,model_v)
    lag=max(0,int(h)-1)
    lrv=np.var(d,ddof=1)
    for k in range(1,lag+1):
        if n<=k: break
        cov=np.cov(d[k:],d[:-k],ddof=1)[0,1]
        lrv += 2*(1-k/(lag+1))*cov
    if not np.isfinite(lrv) or lrv<=0: return np.nan,np.nan
    stat=float(np.mean(d)/np.sqrt(lrv/n))
    p=float(2*student_t.sf(abs(stat),df=n-1))
    return stat,p

In [ ]:
keys=["Currency","Horizon","Year","Forecast_Origin","Realized_Variance"]

wide=cand.pivot_table(
    index=keys,
    columns="Model",
    values="Variance_Forecast",
    aggfunc="first"
).reset_index()

required=["EWMA94","HIST20","HIST60"]
missing=[c for c in required if c not in wide.columns]
if missing:
    raise ValueError(f"Missing required Script 12 baselines: {missing}")

wide["Robust_Anchor"]=wide[required].median(axis=1)
wide["EWMA_Benchmark"]=wide["EWMA94"]

PARAM_MODELS=[c for c in ["GARCH_NORMAL","GARCH_T","EGARCH_T","GJR_GARCH_T"] if c in wide.columns]

print("Parametric models:",PARAM_MODELS)
print("Rows:",len(wide))

In [ ]:
regime_parts=[]

for cur in CURRENCIES:
    mid=pd.to_numeric(df[f"{cur}_mid"],errors="coerce")
    ret=100*np.log(mid/mid.shift(1))
    r2=ret**2

    short_var=r2.rolling(10,min_periods=5).mean()
    long_var=r2.rolling(60,min_periods=20).mean()
    stress_ratio=short_var/np.maximum(long_var,EPS)

    regime_parts.append(pd.DataFrame({
        "Currency":cur,
        "Forecast_Origin":df["Date"],
        "ShortVar10":short_var,
        "LongVar60":long_var,
        "Stress_Ratio":stress_ratio
    }))

regime=pd.concat(regime_parts,ignore_index=True)

wide=wide.merge(
    regime,
    on=["Currency","Forecast_Origin"],
    how="left"
)

print("Missing stress ratio:",wide["Stress_Ratio"].isna().sum())

In [ ]:
def robust_forecast(frame,model,model_weight,stress_threshold,stress_cap):
    raw=pd.to_numeric(frame[model],errors="coerce").to_numpy(float)
    anchor=pd.to_numeric(frame["Robust_Anchor"],errors="coerce").to_numpy(float)
    stress=pd.to_numeric(frame["Stress_Ratio"],errors="coerce").to_numpy(float)

    lower=np.maximum(LOWER_MULT*anchor,EPS)
    upper=np.maximum(UPPER_MULT*anchor,lower)
    clipped=np.clip(raw,lower,upper)

    stressed=np.isfinite(stress)&(stress>stress_threshold)

    effective_weight=np.full(len(frame),model_weight,dtype=float)
    effective_weight[stressed]=np.minimum(model_weight,stress_cap)

    bad=~np.isfinite(clipped)
    effective_weight[bad]=0.0
    clipped[bad]=anchor[bad]

    final=(1-effective_weight)*anchor+effective_weight*clipped

    return np.maximum(final,EPS),stressed,effective_weight,raw,clipped

In [ ]:
strategies=[{
    "Strategy":"ANCHOR_ONLY",
    "Model":"Robust_Anchor",
    "Model_Weight":0.0,
    "Stress_Threshold":np.nan,
    "Stress_Cap":0.0
}]

for model in PARAM_MODELS:
    for w in MODEL_WEIGHTS:
        for threshold in STRESS_THRESHOLDS:
            for cap in STRESS_MODEL_WEIGHT_CAPS:
                if cap>w:
                    continue
                strategies.append({
                    "Strategy":f"{model}_W{w}_T{threshold}_C{cap}",
                    "Model":model,
                    "Model_Weight":w,
                    "Stress_Threshold":threshold,
                    "Stress_Cap":cap
                })

strategy_grid=pd.DataFrame(strategies)

print("Strategies:",len(strategy_grid))
display(strategy_grid.head())

In [ ]:
def year_frame(cur,h,year):
    return wide[
        (wide["Currency"]==cur)
        &(wide["Horizon"]==h)
        &(wide["Year"]==year)
    ].copy().sort_values("Forecast_Origin")

def apply_strategy(frame,row):
    if row["Strategy"]=="ANCHOR_ONLY":
        pred=np.maximum(pd.to_numeric(frame["Robust_Anchor"],errors="coerce").to_numpy(float),EPS)
        stressed=np.zeros(len(frame),dtype=bool)
        ew=np.zeros(len(frame),dtype=float)
        raw=pred.copy()
        clipped=pred.copy()
        return pred,stressed,ew,raw,clipped

    return robust_forecast(
        frame,
        row["Model"],
        float(row["Model_Weight"]),
        float(row["Stress_Threshold"]),
        float(row["Stress_Cap"])
    )

def score_strategy(cur,h,year,row):
    f=year_frame(cur,h,year)
    if f.empty: return np.nan
    pred,*_=apply_strategy(f,row)
    return qlike(f["Realized_Variance"],pred)

In [ ]:
selection_rows=[]
outer_parts=[]
candidate_score_rows=[]

for cur in CURRENCIES:
    for h in HORIZONS:
        print(f"\n===== {cur} H{h} =====")

        for year in YEARS:
            inner_year=year-1
            scores=[]

            for _,s in strategy_grid.iterrows():
                loss=score_strategy(cur,h,inner_year,s)
                if np.isfinite(loss):
                    scores.append({**s.to_dict(),"Inner_QLIKE":loss})

            scoretab=pd.DataFrame(scores)
            if scoretab.empty:
                raise RuntimeError(f"No valid strategies for {cur} H{h} {year}")

            scoretab=scoretab.sort_values(
                ["Inner_QLIKE","Model_Weight","Stress_Cap"],
                ascending=[True,True,True]
            ).reset_index(drop=True)

            chosen=scoretab.iloc[0]
            f=year_frame(cur,h,year)

            pred,stressed,eff_w,raw,clipped=apply_strategy(f,chosen)

            out=f[[
                "Currency","Horizon","Year","Forecast_Origin",
                "Realized_Variance","EWMA_Benchmark","Robust_Anchor","Stress_Ratio"
            ]].copy()

            out=out.rename(columns={"Year":"Validation_Year"})
            out["Regime_Robust_Forecast"]=pred
            out["Stress_Active"]=stressed
            out["Effective_Model_Weight"]=eff_w
            out["Raw_Selected_Model_Forecast"]=raw
            out["Clipped_Selected_Model_Forecast"]=clipped
            out["Selected_Strategy"]=chosen["Strategy"]
            out["Selected_Model"]=chosen["Model"]
            out["Guardrail_Active"]=(
                np.isfinite(raw)&np.isfinite(clipped)&(np.abs(raw-clipped)>1e-12)
            )

            outer_parts.append(out)

            selection_rows.append({
                "Currency":cur,
                "Horizon":h,
                "Outer_Year":year,
                "Inner_Year":inner_year,
                "Selected_Strategy":chosen["Strategy"],
                "Selected_Model":chosen["Model"],
                "Model_Weight":chosen["Model_Weight"],
                "Stress_Threshold":chosen["Stress_Threshold"],
                "Stress_Cap":chosen["Stress_Cap"],
                "Inner_QLIKE":chosen["Inner_QLIKE"]
            })

            scoretab["Currency"]=cur
            scoretab["Horizon"]=h
            scoretab["Outer_Year"]=year
            candidate_score_rows.append(scoretab)

            print(year,"|",chosen["Strategy"],"| inner QLIKE =",round(float(chosen["Inner_QLIKE"]),6))

final_predictions=pd.concat(outer_parts,ignore_index=True)
selections=pd.DataFrame(selection_rows)
candidate_scores=pd.concat(candidate_score_rows,ignore_index=True)

In [ ]:
pooled=[]

for (cur,h),g in final_predictions.groupby(["Currency","Horizon"]):
    m=np.isfinite(g["Realized_Variance"])&np.isfinite(g["Regime_Robust_Forecast"])&np.isfinite(g["EWMA_Benchmark"])
    x=g.loc[m].copy()

    q_model=qlike(x["Realized_Variance"],x["Regime_Robust_Forecast"])
    q_ewma=qlike(x["Realized_Variance"],x["EWMA_Benchmark"])
    mse_model=variance_mse(x["Realized_Variance"],x["Regime_Robust_Forecast"])
    mse_ewma=variance_mse(x["Realized_Variance"],x["EWMA_Benchmark"])
    vr_model=vol_rmse(x["Realized_Variance"],x["Regime_Robust_Forecast"])
    vr_ewma=vol_rmse(x["Realized_Variance"],x["EWMA_Benchmark"])
    dm_stat,dm_p=dm_qlike(x["Realized_Variance"],x["Regime_Robust_Forecast"],x["EWMA_Benchmark"],h)

    pooled.append({
        "Currency":cur,
        "Horizon":h,
        "Observations":len(x),
        "EWMA_QLIKE":q_ewma,
        "Robust_QLIKE":q_model,
        "QLIKE_Gain_vs_EWMA":q_ewma-q_model,
        "EWMA_Variance_MSE":mse_ewma,
        "Robust_Variance_MSE":mse_model,
        "MSE_Skill_vs_EWMA":1-mse_model/mse_ewma if mse_ewma>0 else np.nan,
        "EWMA_Vol_RMSE":vr_ewma,
        "Robust_Vol_RMSE":vr_model,
        "DM_Stat_QLIKE_vs_EWMA":dm_stat,
        "DM_pvalue_QLIKE_vs_EWMA":dm_p,
        "Beats_EWMA_QLIKE":q_model<q_ewma,
        "Stress_Activation_Rate":float(x["Stress_Active"].mean()),
        "Guardrail_Activation_Rate":float(x["Guardrail_Active"].mean())
    })

pooled_results=pd.DataFrame(pooled).sort_values(["Currency","Horizon"])
display(pooled_results)

print("\nTasks beating EWMA on pooled QLIKE:",
      int(pooled_results["Beats_EWMA_QLIKE"].sum()),"/",len(pooled_results))

In [ ]:
yearly=[]

for (cur,h,year),g in final_predictions.groupby(["Currency","Horizon","Validation_Year"]):
    q_model=qlike(g["Realized_Variance"],g["Regime_Robust_Forecast"])
    q_ewma=qlike(g["Realized_Variance"],g["EWMA_Benchmark"])
    dm_stat,dm_p=dm_qlike(g["Realized_Variance"],g["Regime_Robust_Forecast"],g["EWMA_Benchmark"],h)

    yearly.append({
        "Currency":cur,
        "Horizon":h,
        "Validation_Year":year,
        "Observations":len(g),
        "EWMA_QLIKE":q_ewma,
        "Robust_QLIKE":q_model,
        "QLIKE_Gain_vs_EWMA":q_ewma-q_model,
        "Robust_Vol_RMSE":vol_rmse(g["Realized_Variance"],g["Regime_Robust_Forecast"]),
        "DM_Stat_QLIKE_vs_EWMA":dm_stat,
        "DM_pvalue_QLIKE_vs_EWMA":dm_p,
        "Beats_EWMA":q_model<q_ewma,
        "Stress_Activation_Rate":float(g["Stress_Active"].mean()),
        "Guardrail_Activation_Rate":float(g["Guardrail_Active"].mean())
    })

yearly_results=pd.DataFrame(yearly).sort_values(["Currency","Horizon","Validation_Year"])
display(yearly_results)

In [ ]:
def kupiec_uc_test(exceedances,alpha=0.05):
    e=np.asarray(exceedances,dtype=bool)
    n=len(e)
    x=int(e.sum())
    if n==0: return np.nan,np.nan

    phat=x/n
    def slog(p):
        return np.log(np.clip(p,1e-12,1-1e-12))

    logL0=(n-x)*slog(1-alpha)+x*slog(alpha)
    logL1=(n-x)*slog(1-phat)+x*slog(phat)

    lr=-2*(logL0-logL1)
    p=1-stats.chi2.cdf(lr,df=1)
    return float(lr),float(p)

var_rows=[]

for cur in CURRENCIES:
    g=final_predictions[
        (final_predictions["Currency"]==cur)
        &(final_predictions["Horizon"]==1)
    ].copy()

    mid=pd.to_numeric(df[f"{cur}_mid"],errors="coerce")
    ret=100*np.log(mid/mid.shift(1))

    tmp=pd.DataFrame({
        "Forecast_Origin":df["Date"],
        "Next_Return":ret.shift(-1)
    })

    g=g.merge(tmp,on="Forecast_Origin",how="left")

    sigma=np.sqrt(np.maximum(g["Regime_Robust_Forecast"].to_numpy(float),EPS))
    threshold=stats.norm.ppf(0.05)*sigma
    actual=g["Next_Return"].to_numpy(float)

    m=np.isfinite(actual)&np.isfinite(threshold)
    exceed=actual[m]<threshold[m]

    lr,p=kupiec_uc_test(exceed,0.05)

    var_rows.append({
        "Currency":cur,
        "Observations":int(m.sum()),
        "Expected_Exceedance_Rate":0.05,
        "Observed_Exceedance_Rate":float(exceed.mean()) if m.sum() else np.nan,
        "Exceedances":int(exceed.sum()),
        "Kupiec_LR_UC":lr,
        "Kupiec_pvalue":p,
        "Coverage_Accepted_5pct":bool(p>=0.05) if np.isfinite(p) else False
    })

var_results=pd.DataFrame(var_rows)
display(var_results)

In [ ]:
audit=[]

for (cur,h),g in final_predictions.groupby(["Currency","Horizon"]):
    anchor=np.maximum(g["Robust_Anchor"].to_numpy(float),EPS)
    raw=g["Raw_Selected_Model_Forecast"].to_numpy(float)
    final=g["Regime_Robust_Forecast"].to_numpy(float)

    audit.append({
        "Currency":cur,
        "Horizon":h,
        "Raw_Min_Ratio_to_Anchor":float(np.nanmin(raw/anchor)),
        "Raw_Max_Ratio_to_Anchor":float(np.nanmax(raw/anchor)),
        "Final_Min_Ratio_to_Anchor":float(np.nanmin(final/anchor)),
        "Final_Max_Ratio_to_Anchor":float(np.nanmax(final/anchor)),
        "Stress_Activation_Rate":float(g["Stress_Active"].mean()),
        "Guardrail_Activation_Rate":float(g["Guardrail_Active"].mean())
    })

stability_audit=pd.DataFrame(audit)
display(stability_audit)

In [ ]:
final_predictions.to_csv(
    PRED_DIR/"12B_regime_robust_volatility_predictions_2019_2023.csv",
    index=False
)
pooled_results.to_csv(
    TABLE_DIR/"12B_regime_robust_volatility_pooled_results.csv",
    index=False
)
yearly_results.to_csv(
    TABLE_DIR/"12B_regime_robust_volatility_yearly_results.csv",
    index=False
)
selections.to_csv(
    TABLE_DIR/"12B_regime_robust_volatility_selections.csv",
    index=False
)
candidate_scores.to_csv(
    TABLE_DIR/"12B_regime_robust_volatility_candidate_scores.csv",
    index=False
)
var_results.to_csv(
    TABLE_DIR/"12B_regime_robust_volatility_kupiec.csv",
    index=False
)
stability_audit.to_csv(
    TABLE_DIR/"12B_regime_robust_volatility_stability_audit.csv",
    index=False
)

quality_gate=pd.DataFrame([
    {"Area":"Holdout isolation","Status":"PASS","Evidence":"No 2024–2025 observations loaded"},
    {"Area":"Candidate models","Status":"PASS","Evidence":"Uses only Script 12 model forecasts and historical baselines"},
    {"Area":"Current-year leakage","Status":"PASS","Evidence":"Strategy selected from previous-year QLIKE only"},
    {"Area":"Regime signal","Status":"PASS","Evidence":"10-day / 60-day variance ratio uses information available at origin"},
    {"Area":"Stability control","Status":"PASS","Evidence":"Two-sided bounds use only contemporaneous robust anchor"},
    {"Area":"Primary metric","Status":"PASS","Evidence":"QLIKE against EWMA on matched observations"},
    {"Area":"Risk calibration","Status":"PASS","Evidence":"Kupiec unconditional-coverage test added for H1 VaR"}
])

display(quality_gate)

quality_gate.to_csv(
    TABLE_DIR/"12B_regime_robust_volatility_quality_gate.csv",
    index=False
)

print("\nSCRIPT 12B COMPLETE")
print("Tasks beating EWMA on pooled QLIKE:",
      int(pooled_results["Beats_EWMA_QLIKE"].sum()),"/",len(pooled_results))
print("2024–2025 remains untouched.")

## Decision rule after running

Do not retain Script 12B just because it has more wins.

Keep it only if:
- USD H3/H7 instability is materially reduced;
- QLIKE remains competitive or improves;
- variance MSE / volatility RMSE are not pathological;
- guardrails are not active almost all the time;
- VaR coverage remains reasonable;
- any significance claim is supported by the QLIKE loss-differential test.

This keeps the volatility framework accurate and defensible.